<link rel="stylesheet" href="/site-assets/css/gemma.css">
<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Google+Symbols:opsz,wght,FILL,GRAD@20..48,100..700,0..1,-50..200" />

##### Copyright 2024 Google LLC。

In [1]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Keras CodeGemma 快速入門

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/gemma/docs/codegemma/keras_quickstart"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td> <td>    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/docs/codegemma/keras_quickstart.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/docs/codegemma/keras_quickstart.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fdocs%2Fcodegemma%2Fkeras_quickstart.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/docs/codegemma/keras_quickstart.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

CodeGemma 是一系列輕量級、最先進的開放模型，採用與創建 Gemini 模型相同的研究和技術而構建。
CodeGemma 模型使用超過 5000 億個 tokens 主要程式碼進行訓練
與 Gemma 型號系列具有相同的架構。因此，CodeGemma 模型在完成和完成方面都實現了最先進的程式碼效能
和發電任務，同時保持強勁
大規模的理解和推論能力。
CodeGemma 有 3 種變體：
* 7B 程式碼預訓練模型
* 7B 指令調整的程式碼模型
* 一個 2B 模型，專門針對程式碼填充和開放式生成進行訓練。

本指南將引導您使用 CodeGemma 2B 模型和 KerasHub 來完成程式碼完成任務。

## 設定

### 訪問CodeGemma

要完成本教學，您首先需要完成 [Gemma 設定](https://ai.google.dev/gemma/docs/setup) 中的設定說明。 Gemma 設定說明向您展示如何執行以下操作：
* 在 [kaggle.com](https://kaggle.com){:.external} 上造訪Gemma。
* 選擇具有足夠資源執行的 Colab runtime
Gemma 2B 型號。* 產生並設定 Kaggle 使用者名稱和 API 金鑰。

完成 Gemma 設定後，請前往下一部分，您將為 Colab 環境設定環境變數。

### 選擇runtime

要完成本教學，您需要擁有 Colab runtime 以及足夠的資源來執行 CodeGemma 2B 模型。在這種情況下，您可以使用 T4 GPU：
1. 在Colab視窗的右上角，選擇&#9662; （**附加連線選項**）。
2. 選擇**更改 runtime 類型**。
3. 在 **硬體加速器** 下，選擇 **T4 GPU**。

### 設定您的 API 金鑰

若要使用 Gemma，您必須提供 Kaggle 使用者名稱和 Kaggle API 金鑰。
若要產生 Kaggle API 金鑰，請前往 Kaggle 使用者個人資料的 **帳戶** 選項卡，然後選擇 **建立新 token**。這將觸發包含您的 API 憑證的 `kaggle.json` 檔案的下載。
在 Colab 中，選擇左側窗格中的 **Secrets** (🔑)，然後新增您的 Kaggle 使用者名稱和 Kaggle API 金鑰。將您的使用者名稱儲存在名稱`KAGGLE_USERNAME` 下，將您的API 金鑰儲存在名稱`KAGGLE_KEY` 下。

### 設定環境變數

設定`KAGGLE_USERNAME` 和`KAGGLE_KEY` 的環境變數。

In [2]:
import os
from google.colab import userdata

os.environ["KAGGLE_USERNAME"] = userdata.get('KAGGLE_USERNAME')
os.environ["KAGGLE_KEY"] = userdata.get('KAGGLE_KEY')

### 安裝依賴項

In [3]:
!pip install -q -U keras-hub
!pip install -q -U keras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 792.1/792.1 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 53.2 MB/s eta 0:00:00


### 選擇後端

Keras 是高級、多framework 深度學習API，設計簡單易用。使用Keras 3，您可以在三個後端之一上執行工作流程：TensorFlow、JAX 或PyTorch。
在本教學中，設定 TensorFlow 的後端。

In [4]:
os.environ["KERAS_BACKEND"] = "tensorflow"  # Or "jax" or "torch".

### 導入包

導入 Keras 和 KerasHub。

In [5]:
import keras_hub
import keras

# Run at half precision.
keras.config.set_floatx("bfloat16")

### 負載模型

KerasHub 提供了許多流行的[模型架構](https://keras.io/api/keras_nlp/models/){:.external} 的實作。在本教學中，您將使用 `GemmaCausalLM` 建立模型，這是用於因果語言建模的端對端 Gemma 模型。因果語言模型根據前一個 tokens 預測下一個 token。
使用 `from_preset` 方法建立模型：

In [7]:
gemma_lm = keras_hub.models.GemmaCausalLM.from_preset("code_gemma_2b_en")
gemma_lm.summary()

100%|██████████| 785/785 [00:00<00:00, 1.64MB/s]


100%|██████████| 4.67G/4.67G [00:50<00:00, 99.2MB/s]


100%|██████████| 591/591 [00:00<00:00, 946kB/s]


100%|██████████| 4.04M/4.04M [00:00<00:00, 43.1MB/s]


Preprocessor: "gemma_causal_lm_preprocessor"

...

Model: "gemma_causal_lm"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ gemma_backbone                │ (None, None, 2048)        │   2,506,172,416 │ padding_mask[0][0],        │
│ (GemmaBackbone)               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_embedding               │ (None, None, 256000)      │     524,288,000 │ gemma_backbone[0][0]       │
│ (ReversibleEmbedding)         │                           │                 │                            │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 2,506,172,416 (4.67 GB)

 Trainable params: 2,506,172,416 (4.67 GB)

 Non-trainable params: 0 (0.00 B)

`from_preset` 方法根據預設的架構和權重實例化模型。在上面的程式碼中，字串`code_gemma_2b_en`指定了預設架構－具有20億個參數的CodeGemma模型。
注意： CodeGemma 型號有 7
十億個參數也可用。要在 Colab 中執行更大的模型，您需要存取付費計劃中提供的高級 GPU。

## 中間填寫代碼補全

此範例使用 CodeGemma 的中間填充 (FIM) 功能來根據周圍上下文完成程式碼。這在程式碼編輯器應用程式中特別有用，用於插入程式碼，其中文字遊標基於其周圍的程式碼（遊標之前和之後）。

CodeGemma 允許您使用 4 個使用者定義的 tokens - 3 個用於 FIM 和一個 `<|file_separator|>` token 用於多檔案上下文支援。使用它們來定義常數。

In [8]:
BEFORE_CURSOR = "<|fim_prefix|>"
AFTER_CURSOR = "<|fim_suffix|>"
AT_CURSOR = "<|fim_middle|>"
FILE_SEPARATOR = "<|file_separator|>"

定義模型的停止tokens。

In [9]:
END_TOKEN = gemma_lm.preprocessor.tokenizer.end_token

stop_tokens = (BEFORE_CURSOR, AFTER_CURSOR, AT_CURSOR, FILE_SEPARATOR, END_TOKEN)

stop_token_ids = tuple(gemma_lm.preprocessor.tokenizer.token_to_id(x) for x in stop_tokens)

格式化 prompt 以完成程式碼。注意：* 任何 FIM tokens 與前綴和後綴之間不應有空格
* FIM 中間 token 應該位於最後，以便模型繼續填充
* 前綴或後綴可以為空，具體取決於遊標目前在文件中的位置，或者您想要為模型提供多少上下文


使用輔助函數格式化prompt。

In [10]:
def format_completion_prompt(before, after):
    return f"{BEFORE_CURSOR}{before}{AFTER_CURSOR}{after}{AT_CURSOR}"

before = "import "
after = """if __name__ == "__main__":\n    sys.exit(0)"""
prompt = format_completion_prompt(before, after)
print(prompt)

<|fim_prefix|>import <|fim_suffix|>if __name__ == "__main__":
    sys.exit(0)<|fim_middle|>


執行prompt。建議串流回應tokens。遇到任何使用者定義的或回合結束/句子tokens時停止streaming以獲得結果代碼完成。

In [11]:
gemma_lm.generate(prompt, stop_token_ids=stop_token_ids, max_length=128)

'<|fim_prefix|>import <|fim_suffix|>if __name__ == "__main__":\n    sys.exit(0)<|fim_middle|>sys\n<|file_separator|>'

該模型提供 `sys` 作為建議的程式碼完成。

## 概括

本教學引導您使用 CodeGemma 根據周圍上下文填充程式碼。接下來，請查看[使用 CodeGemma 和 KerasHub notebook 進行 AI 輔助編程](https://ai.google.dev/gemma/docs/codegemma/code_assist_keras)，以了解有關如何使用 CodeGemma 的更多範例。
另請參閱 [CodeGemma 型號卡](https://ai.google.dev/gemma/docs/codegemma/model_card) 以了解 CodeGemma 型號的技術規格。